# Ejemplo Práctico Resuelto: AHP vs. BWM
## Sesión 2: Ponderación Subjetiva de Criterios en Cadena de Suministro

Este notebook implementa el cálculo de pesos para criterios de decisión usando dos metodologías fundamentales:
1. **Proceso de Jerarquía Analítica (AHP)** (Saaty)
2. **Best-Worst Method (BWM)** (Rezaei)

### El Escenario Logístico
Un Centro de Distribución (CEDIS) de e-commerce en Monterrey necesita contratar un transportista de carga consolidada nacional (LTL) evaluando 4 criterios:
*   **$C_1$ (Costo):** Tarifa por kilómetro.
*   **$C_2$ (Confiabilidad):** Exactitud de entrega (OTIF).
*   **$C_3$ (Visibilidad):** Rastreo GPS en tiempo real.
*   **$C_4$ (Sustentabilidad):** Emisiones y flota ecológica.

## 1. Proceso de Jerarquía Analítica (AHP)

En AHP, el decisor realiza las comparaciones pareadas de los 4 criterios utilizando la escala fundamental de 1 a 9 de Saaty. Esto genera una matriz recíproca positiva:
$$\mathbf{A} = \begin{pmatrix}
1 & 1/2 & 2 & 3 \\
2 & 1 & 3 & 6 \\
1/2 & 1/3 & 1 & 2 \\
1/3 & 1/6 & 1/2 & 1
\end{pmatrix}$$

### Pasos para resolver en Python:
1.  **Definir la matriz de comparaciones.**
2.  **Calcular el autovalor máximo ($\lambda_{\max}$) y su autovector asociado** (que representa los pesos).
3.  **Normalizar el autovector** para que la suma sea 1.0.
4.  **Calcular el Índice de Consistencia ($CI$) y la Razón de Consistencia ($CR$)** para verificar la validez de los juicios.

In [5]:
import numpy as np

# 1. Definir la matriz de comparaciones pareadas AHP (recíproca)
A = np.array([
    [1.0,   1/2, 2.0, 3.0],
    [2.0,   1.0, 3.0, 6.0],
    [1/2,   1/3, 1.0, 2.0],
    [1/3,   1/6, 1/2, 1.0]
])

print("Matriz AHP (A):")
print(A)

# 2. Calcular autovalores y autovectores usando álgebra lineal de NumPy
eigenvalues, eigenvectors = np.linalg.eig(A)

# El autovalor principal es el máximo número real
lambda_max = np.real(eigenvalues.max())

# El autovector asociado al autovalor máximo
max_index = np.argmax(np.real(eigenvalues))
w_eigen = np.real(eigenvectors[:, max_index])

# 3. Normalizar el autovector para que los pesos sumen 1
w_ahp = w_eigen / np.sum(w_eigen)

print(f"\nAutovalor máximo (lambda_max): {lambda_max:.4f}")
print("Pesos calculados (AHP):")
criterios = ["Costo (C1)", "Confiabilidad (C2)", "Visibilidad (C3)", "Sustentabilidad (C4)"]
for crit, peso in zip(criterios, w_ahp):
    print(f"  * {crit}: {peso:.4f}")

Matriz AHP (A):
[[1.         0.5        2.         3.        ]
 [2.         1.         3.         6.        ]
 [0.5        0.33333333 1.         2.        ]
 [0.33333333 0.16666667 0.5        1.        ]]

Autovalor máximo (lambda_max): 4.0104
Pesos calculados (AHP):
  * Costo (C1): 0.2672
  * Confiabilidad (C2): 0.4959
  * Visibilidad (C3): 0.1542
  * Sustentabilidad (C4): 0.0827


### Análisis de Consistencia en AHP
Calculamos:
$$CI = \frac{\lambda_{\max} - n}{n-1}$$
$$CR = \frac{CI}{RI}$$

Donde para $n=4$, el Índice Aleatorio ($RI$) es $0.90$. Si $CR < 0.10$ ($10\%$) la consistencia es aceptable.

In [6]:
n = A.shape[0]

# Calcular el Índice de Consistencia (CI)
CI = (lambda_max - n) / (n - 1)

# Índice aleatorio (RI) para n = 4
RI = 0.90

# Calcular la Razón de Consistencia (CR)
CR = CI / RI

print(f"Índice de Consistencia (CI): {CI:.4f}")
print(f"Razón de Consistencia (CR): {CR:.4f} ({CR*100:.2f}%)")

if CR < 0.10:
    print("El decisor es consistente (CR < 10%). Los pesos son válidos.")
else:
    print("El decisor es inconsistente (CR >= 10%). Se deben revisar los juicios.")

Índice de Consistencia (CI): 0.0035
Razón de Consistencia (CR): 0.0038 (0.38%)
El decisor es consistente (CR < 10%). Los pesos son válidos.


### Formulación del PPL de BWM Lineal:
$$\begin{aligned}
\min \quad & \xi^L \\
\text{s.t.} \quad & |w_B - a_{Bj} w_j| \le \xi^L, \quad \forall j \\
& |w_j - a_{jW} w_W| \le \xi^L, \quad \forall j \\
& \sum w_j = 1, \quad w_j \ge 0
\end{aligned}$$

Para implementarlo en Python de forma clara y declarativa, utilizaremos la biblioteca `PuLP`. Esto permite plantear las variables y restricciones algebraicas directamente en el código sin tener que desglosar y construir manualmente la matriz de coeficientes como lo requeriría `scipy.optimize.linprog`.

In [7]:
import pulp

# Definición de parámetros del problema
best_idx = 1   # C2 (0-indexed)
worst_idx = 3  # C4 (0-indexed)

# Vectores de entrada
a_B = [2.0, 1.0, 3.0, 6.0]  # Best-to-Others
a_W = [3.0, 6.0, 2.0, 1.0]  # Others-to-Worst

# 1. Crear el problema de optimización lineal (Minimizar)
prob = pulp.LpProblem("BWM_Lineal", pulp.LpMinimize)

# 2. Definir variables de decisión (pesos w_j e inconsistencia xi_L)
w = [pulp.LpVariable(f"w_{i}", lowBound=0.0, upBound=1.0) for i in range(4)]
xi = pulp.LpVariable("xi_L", lowBound=0.0)

# 3. Función objetivo: Minimizar xi_L
prob += xi

# 4. Restricción de suma: w1 + w2 + w3 + w4 = 1
prob += pulp.lpSum(w) == 1.0

# 5. Restricciones de desviación absoluta (|w_B - a_Bj * w_j| <= xi_L y |w_j - a_jW * w_W| <= xi_L)
for j in range(4):
    # Restricciones de desviación Best-to-Others
    prob += w[best_idx] - a_B[j] * w[j] <= xi
    prob += -(w[best_idx] - a_B[j] * w[j]) <= xi
    
    # Restricciones de desviación Others-to-Worst
    prob += w[j] - a_W[j] * w[worst_idx] <= xi
    prob += -(w[j] - a_W[j] * w[worst_idx]) <= xi

# 6. Resolver el modelo de optimización
prob.solve(pulp.PULP_CBC_CMD(msg=False))

# Extraer resultados
w_bwm = [pulp.value(w_var) for w_var in w]
xi_L = pulp.value(xi)

print("Pesos calculados (BWM con PuLP):")
for crit, peso in zip(criterios, w_bwm):
    print(f"  * {crit}: {peso:.4f}")
print(f"\nDesviación óptima (xi_L): {xi_L:.4f}")

Pesos calculados (BWM con PuLP):
  * Costo (C1): 0.2500
  * Confiabilidad (C2): 0.5000
  * Visibilidad (C3): 0.1667
  * Sustentabilidad (C4): 0.0833

Desviación óptima (xi_L): 0.0000


### Razón de Consistencia en BWM
Se calcula como:
$$CR = \frac{\xi^L}{CI}$$

Donde para un valor de $a_{BW} = 6$ (Confiabilidad vs. Sustentabilidad), el valor de Consistencia Mínima ($CI$) es $3.00$ según la tabla de Rezaei.

In [8]:
# Tabla de CI de Rezaei para BWM
tabla_CI = {1: 0.0, 2: 0.44, 3: 1.0, 4: 1.63, 5: 2.30, 6: 3.0, 7: 3.73, 8: 4.47, 9: 5.23}

a_BW = a_B[worst_idx] # Preferencia del Best sobre el Worst (a_24 = 6)
CI_bwm = tabla_CI[a_BW]

CR_bwm = xi_L / CI_bwm if CI_bwm > 0 else 0.0

print(f"Índice de Consistencia BWM (CI): {CI_bwm:.2f}")
print(f"Razón de Consistencia BWM (CR): {CR_bwm:.4f} ({CR_bwm*100:.2f}%)")

Índice de Consistencia BWM (CI): 3.00
Razón de Consistencia BWM (CR): 0.0000 (0.00%)


## 3. Comparativa de Resultados

Comparemos el vector de pesos obtenido por ambos métodos:

In [9]:
import pandas as pd

df_comp = pd.DataFrame({
    'Criterio': criterios,
    'Pesos AHP': w_ahp,
    'Pesos BWM': w_bwm,
    'Diferencia Absoluta': np.abs(w_ahp - w_bwm)
})

print(df_comp.to_string(index=False))

            Criterio  Pesos AHP  Pesos BWM  Diferencia Absoluta
          Costo (C1)   0.267196   0.250000             0.017196
  Confiabilidad (C2)   0.495939   0.500000             0.004061
    Visibilidad (C3)   0.154209   0.166667             0.012458
Sustentabilidad (C4)   0.082657   0.083333             0.000677
